In [ ]:
%matplotlib widget
# Add the directory containing the package to sys.path
import sys, os
package_dir = os.path.abspath("C:/Users/froll/Documents/Labo/Projets/Outils/swd")
if package_dir not in sys.path:
    sys.path.insert(0, package_dir)
    
from swd import spherical_processing as sp
import importlib

import numpy as np
from numpy import pi
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore")
from tqdm.auto import tqdm
np.set_printoptions(precision=2, suppress=True)

# Enable LaTeX rendering
plt.rc('text', usetex=True)
# Improve resolution
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 300

# Import utils_SH_selection
scripts_dir = os.path.abspath("C:/Users/froll/Documents/Labo/Projets/Violon/scripts")
if scripts_dir not in sys.path:
    sys.path.insert(0, scripts_dir)

import utils_SH_selection
importlib.reload(utils_SH_selection)

In [ ]:
#Vitesse du son au moment de la mesure, dependant de la temperature:
Tc = 21.5 
C = np.sqrt( 1.4 * 287 *(Tc + 273) )
Path = './'
NbMems = 256
NbViolTot = 6
NbViol = 6

NumViolon = np.load('./../results/NumViolon.npz')['NumViolon']
XYZViolins = np.load('./../results/XYZViolinsAligned.npy')

NbMics = 256
NbSrcs = XYZViolins.shape[0] - NbMics

XYZHammerImpact = XYZViolins[NbSrcs-1]
XYZ = XYZViolins-XYZHammerImpact
XYZs = XYZ[:NbSrcs-1]
XYZm = XYZ[NbSrcs:]
XYZHammerImpact = XYZ[NbSrcs-1]

Rm = np.linalg.norm(XYZm,axis=1)
Rmin = np.min(Rm)

### Array Quality Estimation

In [ ]:
from ArrayPerfs import *
print(process_array_quality(XYZm, 1000))

In [ ]:
Pm = np.load('./../results/ViolinsFRFsAndRIs.npz')['Hhm']
frq = np.load('./../results/ViolinsFRFsAndRIs.npz')['frq']

MAX_ORDER = 15
NbSHMax = (MAX_ORDER+1)**2

frqmin = 100
frqMax = 12500

iBand = np.where((frq>=frqmin) & (frq<=frqMax))[0]
Band = frq[iBand]

Dyn = 36
CyclicScale = 'icefire' #edge, icefire, phase, hsv
RealScale = 'seismic'
Magnitudescale = 'inferno'

## Process optimal $\lambda$ and SH truncation order at all frequencies for each Violin

In [ ]:
H_max = sp.compute_SphericalWavesbasis_origin_to_field(XYZm, Band, MAX_ORDER, SH_center=np.array([0,0,0]))

In [ ]:
Process_Opt_Lbda = True # Set to False to load previously computed parameters instead of re-optimizing

target_freqs = Band # Use the actual frequencies selected
results_freqs = list(target_freqs)

if Process_Opt_Lbda:
    from joblib import Parallel, delayed
    import contextlib
    import joblib
    import sys
    
    # Context manager to patch joblib to report into tqdm progress bar
    @contextlib.contextmanager
    def tqdm_joblib(tqdm_object):
        class TqdmBatchCompletionCallback(joblib.parallel.BatchCompletionCallBack):
            def __init__(self, *args, **kwargs):
                super().__init__(*args, **kwargs)
            def __call__(self, *args, **kwargs):
                tqdm_object.update(n=self.batch_size)
                return super().__call__(*args, **kwargs)
        old_batch_callback = joblib.parallel.BatchCompletionCallBack
        joblib.parallel.BatchCompletionCallBack = TqdmBatchCompletionCallback
        try:
            sys.stdout.flush()
            yield tqdm_object
        finally:
            joblib.parallel.BatchCompletionCallBack = old_batch_callback
            tqdm_object.close()

    # Initialize storage
    results_lambdas = {v: [] for v in range(NbViol)}

    print(f"Optimizing lambdas for {NbViol} violins over {len(target_freqs)} frequencies...")

    # 1. Compute optimal lambdas for all frequencies and violins in parallel
    def compute_lambda_for_freq(i, target_freq):
        lambdas_freq = []
        H_max_single = H_max[:, :, i]
        if H_max_single.ndim == 2:
            H_max_single = H_max_single[:, :, np.newaxis]
            
        f_single = np.atleast_1d(target_freq)
        
        for v in range(NbViol):
            p_snapshot = Pm[v, iBand, :][i, :]
            P_single = p_snapshot[:, np.newaxis]
            
            try:
                opt_lambdas = utils_SH_selection.compute_optimal_lambda_gcv(
                    P_single, H_max_single, f_single, lambda_grid=np.logspace(-8, 0, 100)
                )
                lambdas_freq.append(opt_lambdas[0])
            except Exception as e:
                lambdas_freq.append(1e-4) # Fallback lambda
                
        return lambdas_freq

    # Parallelize computing lambdas over frequencies
    print("Computing optimal lambdas...")
    with tqdm_joblib(tqdm(desc="Frequencies", total=len(target_freqs))):
        lambdas_all_freqs = Parallel(n_jobs=-1)(
            delayed(compute_lambda_for_freq)(i, freq) for i, freq in enumerate(target_freqs)
        )
    
    # Store lambdas
    for i in range(len(target_freqs)):
        for v in range(NbViol):
            results_lambdas[v].append(lambdas_all_freqs[i][v])

else:
    results_freqs = np.load(f'./../results/Optimal_Lambdas_Orders_O{MAX_ORDER}.npz', allow_pickle=True)['freqs']
    results_lambdas = np.load(f'./../results/Optimal_Lambdas_Orders_O{MAX_ORDER}.npz', allow_pickle=True)['lambdas'].item()
    print(f"Loaded lambdas parameters from ./../results/Optimal_Lambdas_Orders_O{MAX_ORDER}.npz")

# --- Outlier Filtering for Lambdas ---
import scipy.signal
results_freqs_arr = np.array(results_freqs)

for v in range(NbViol):
    lambdas = np.array(results_lambdas[v])
    log_lambdas = np.log10(lambdas)
    
    # Calculate moving median to identify trend (kernel_size should be odd)
    trend = scipy.signal.medfilt(log_lambdas, kernel_size=51)
    
    # Identify outliers (e.g., threshold of 1.0 in log10 scale = 1 order of magnitude)
    outliers = np.abs(log_lambdas - trend) > 1.0
    
    if np.any(outliers):
        valid = ~outliers
        if np.any(valid):
            # Linear interpolation in log-log space vs frequency
            interp_log = np.interp(np.log10(results_freqs_arr[outliers]), np.log10(results_freqs_arr[valid]), log_lambdas[valid])
            log_lambdas[outliers] = interp_log
            results_lambdas[v] = list(10**log_lambdas)
    else:
        results_lambdas[v] = list(lambdas)

# --- Plotting Results for Lambdas ---
fig, ax = plt.subplots(1, 1, figsize=(10, 3))

try:
    viol_names = NumViolon if 'NumViolon' in locals() else [f"Violin {v}" for v in range(NbViol)]
except:
    viol_names = [f"Violin {v}" for v in range(NbViol)]

cm_plot = plt.get_cmap('gist_rainbow') 
colors = [cm_plot(1.*i/NbViol) for i in range(NbViol)]

for v in range(NbViol):
    lbl = viol_names[v] if v < len(viol_names) else f"Violin {v}"
    ax.loglog(results_freqs, results_lambdas[v], '-', linewidth=1.5, label=lbl, color=colors[v])

ax.set_xlim(frqmin, frqMax)
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Regularization Parameter (GCV)')
ax.set_title('Optimal Regularization Parameter (GCV)')
ax.grid(True, which='both', alpha=0.8)
ax.legend(ncol=2, fontsize='small')

plt.tight_layout()
plt.show()

In [ ]:
Process_Opt_N = True # Set to False to load previously computed parameters instead of re-optimizing

if Process_Opt_N:
    # --- Optimal Order Selection (Multi-Frequency) for ALL Violins ---
    results_orders = {v: [] for v in range(NbViol)}
    last_optimal_orders = {v: 0 for v in range(NbViol)}

    print(f"Optimizing orders for {NbViol} violins over {len(target_freqs)} frequencies...")

    for v in tqdm(range(NbViol), desc="Violins", position=0):
        orders_violin = []
        last_ord = 0       
        for i in tqdm(range(len(target_freqs)), desc=f"Frequency", position=1):
            f_single = np.atleast_1d(target_freqs[i])
            P_single = Pm[v, iBand, :][i, :]
            P_single = P_single[:, np.newaxis]
            best_lambda = results_lambdas[v][i]
            print(f"Violin {v}, Frequency {f_single[0]:.1f} Hz, Lambda: {best_lambda:.2e}\r", end="")
            H_max_single = H_max[:, :, i]
            if H_max_single.ndim == 2:
                H_max_single = H_max_single[:, :, np.newaxis]
                        
            orders = np.arange(1, MAX_ORDER + 1)
            indices = np.arange(NbMics)
            k_folds = 5
            np.random.seed(42) # For reproducibility
            np.random.shuffle(indices)
            folds = np.array_split(indices, k_folds)
            cv_errors = []
            for order in orders:
                NbSH = (order + 1)**2
                # Extraction de la matrice de transfert tronquée pour cet ordre
                H_order = H_max_single[:, :NbSH, :] 
                fold_errors = []
                for fold in folds:
                    train_idx = np.setdiff1d(indices, fold)
                    test_idx = fold
                    
                #   Split Train/Test
                    H_train = H_order[train_idx, :, :]
                    P_train = P_single[train_idx, :]
                    
                    # Résolution sur l'ensemble d'entraînement avec sp.compute_SHcoefs
                    Cmn_train = sp.compute_SHcoefs(pmeas=P_train, H=H_train, lambda_reg=best_lambda)
                    
                    # Prédiction sur l'ensemble de test
                    H_test = H_order[test_idx, :, 0]
                    P_test_pred = H_test @ Cmn_train[:, 0]
                    
                    P_test = P_single[test_idx, 0]
                    
                    # Erreur de validation croisée
                    err = np.linalg.norm(P_test - P_test_pred)**2 / np.linalg.norm(P_test)**2
                    fold_errors.append(err)
                    
                cv_errors.append(np.mean(fold_errors))

            cv_errors = np.array(cv_errors)
            
            # Prise en compte de la contrainte de monotonie (ordre >= dernier ordre choisi)
            valid_mask = orders >= last_ord
            if np.any(valid_mask):
                valid_orders = orders[valid_mask]
                valid_cv_errors = cv_errors[valid_mask]
                best_idx_local = np.argmin(valid_cv_errors)
                best_OSH = int(valid_orders[best_idx_local])
            else:
                best_OSH = int(orders[np.argmin(cv_errors)])

            last_ord = best_OSH
            orders_violin.append(best_OSH)
            
        results_orders[v] = orders_violin



Optimizing orders for 6 violins over 2481 frequencies...


Violins:   0%|          | 0/6 [00:00<?, ?it/s]

In [ ]:
            
                
#                 fold_errors = []
#                 for fold in folds:
#                     train_idx = np.setdiff1d(indices, fold)
#                     test_idx = fold
                    
#                     # Split Train/Test
#                     H_train = H_order[train_idx, :, :]
#                     P_train = P_single[train_idx, :]
                    
#                     # Résolution sur l'ensemble d'entraînement avec sp.compute_SHcoefs
#                     Cmn_train = sp.compute_SHcoefs(pmeas=P_train, H=H_train, lambda_reg=best_lambda)
                    
#                     # Prédiction sur l'ensemble de test
#                     H_test = H_order[test_idx, :, 0]
#                     P_test_pred = H_test @ Cmn_train[:, 0]
                    
#                     P_test = P_single[test_idx, 0]
                    
#                     # Erreur de validation croisée
#                     err = np.linalg.norm(P_test - P_test_pred)**2 / np.linalg.norm(P_test)**2
#                     fold_errors.append(err)
                    
#                 cv_errors.append(np.mean(fold_errors))

#             cv_errors = np.array(cv_errors)
            
#             # Prise en compte de la contrainte de monotonie (ordre >= dernier ordre choisi)
#             valid_mask = orders >= last_ord
#             if np.any(valid_mask):
#                 valid_orders = orders[valid_mask]
#                 valid_cv_errors = cv_errors[valid_mask]
#                 best_idx_local = np.argmin(valid_cv_errors)
#                 best_OSH = int(valid_orders[best_idx_local])
#             else:
#                 best_OSH = int(orders[np.argmin(cv_errors)])

#             last_ord = best_OSH
#             orders_violin.append(best_OSH)
            
#         results_orders[v] = orders_violin


#     # Save optimized lambdas and SH orders first
#     np.savez(f'./../results/Optimal_Lambdas_Orders_O{MAX_ORDER}.npz', 
#             orders=results_orders, 
#             lambdas=results_lambdas, 
#             freqs=results_freqs)
#     print(f"Saved optimization parameters to ./../results/Optimal_Lambdas_Orders_O{MAX_ORDER}.npz")

# else:
#     results_orders = np.load(f'./../results/Optimal_Lambdas_Orders_O{MAX_ORDER}.npz', allow_pickle=True)['orders'].item()
#     print(f"Loaded order parameters from ./../results/Optimal_Lambdas_Orders_O{MAX_ORDER}.npz")

# # --- Plotting Results for Orders ---
# fig, ax = plt.subplots(1, 1, figsize=(6, 3))

# try:
#     viol_names = NumViolon if 'NumViolon' in locals() else [f"Violin {v}" for v in range(NbViol)]
# except:
#     viol_names = [f"Violin {v}" for v in range(NbViol)]

# cm_plot = plt.get_cmap('gist_rainbow') 
# colors = [cm_plot(1.*i/NbViol) for i in range(NbViol)]

# for v in range(NbViol):
#     lbl = viol_names[v] if v < len(viol_names) else f"Violin {v}"
#     ax.semilogx(results_freqs, results_orders[v], '-', linewidth=1.5, label=lbl, color=colors[v])


# ax.set_xlim(frqmin, frqMax)
# ax.set_xlabel('Frequency (Hz)')
# ax.set_ylabel('Optimal SH Order')
# ax.set_title('Optimal SH Truncation Order (CV with Monotonicity)')
# ax.grid(True, which='both', alpha=0.8)
# ax.set_yticks(np.arange(0, MAX_ORDER + 2, 1))
# ax.legend(ncol=2, fontsize='small')

# plt.tight_layout()
# plt.show()
# plt.savefig(os.path.join(Path, './../results/Optimal_Lambdas_and_SH_Orders.png'), dpi=300)

In [ ]:

print(f"\nComputing Cmn for all {NbViol} violins using compute_SHcoefs_VariableLbdas...")

NbSH_max = (MAX_ORDER + 1)**2
Cmn_all = np.zeros((NbViol, NbSH_max, len(results_freqs)), dtype=complex)

kvect = 2 * np.pi * np.array(results_freqs) / C
# Compute basis up to MAX_ORDER for all frequencies at once
H_max = sp.compute_SphericalWavesbasis_origin_to_field(
    XYZm, kvect, MAX_ORDER, SH_center=np.array([0,0,0])
)

for v in tqdm(range(NbViol), desc="Processing violins"):
    order_v = np.array(results_orders[v], dtype=int)
    lam_v = np.array(results_lambdas[v])
    
    # pmeas should be (N_fieldpoints, N_k)
    pmeas_v = Pm[v, iBand, :].T 
    
    # Use swd2 dedicated function
    c_coeffs = sp.compute_SHcoefs_VariableLbdas(
        pmeas=pmeas_v, 
        H=H_max, 
        N_SH_vect=order_v, 
        lambda_reg=lam_v
    )
    
    Cmn_all[v, :, :] = c_coeffs

# Update Globals for consistency (Set Cmn0 to the selected violin NumV)
NumV = 1
Cmn0 = Cmn_all[NumV]
OSH = MAX_ORDER
O_SH_vect = np.array(results_orders[NumV])
print(f"Cmn computation complete. Shape: {Cmn_all.shape}")

In [ ]:
np.savez('./../results/CmnO'+str(MAX_ORDER)+'_Optimization_Results.npz', 
         orders=results_orders, 
         lambdas=results_lambdas, 
         freqs=results_freqs,
         Cmn_all=Cmn_all)
print("Saved detailed optimization results to ./../results/CmnO"+str(MAX_ORDER)+"_Optimization_Results.npz")